# UniCell Automated Single-Cell Annotation

This notebook uses a trained UniCell model to predict the following for single-cell data:

- Cell type (Cell Ontology ID)
- Tissue of origin
- Species

## How to Use This Notebook

1. Run each cell in order using **Run** in the Jupyter menu.
2. Usually, only “Step 2: Specify Paths and Parameters” needs to be modified.
3. After inference finishes, the annotated data is saved as a new `.h5ad` file; the original input is not overwritten.

> **Data requirements**: The input must be an AnnData `.h5ad` file, and `adata.var_names` should contain gene symbols. The expression matrix should be preprocessed in the same way as the model's training data. For datasets containing millions of cells, use the project's chunked-inference notebook to avoid running out of memory.

## Step 1: Load the Runtime Environment

If a required Python package is reported missing here, contact the model maintainer to configure the UniCell runtime environment.

In [ ]:
from pathlib import Path
import json
import time

import anndata as ad
import numpy as np
import pandas as pd
import torch

from unicell.anno_predict import unicell_predict
from unicell.utils.utils import compute_metrics as compute_metrics_cls

print("Runtime environment loaded successfully")
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Step 2: Specify Paths and Parameters

Usually, only `INPUT_H5AD` and `OUTPUT_H5AD` need to be modified.

- `INPUT_H5AD`: The single-cell dataset to annotate.
- `OUTPUT_H5AD`: The location at which to save the prediction results. Do not use the same path as the input file.
- `MODEL_DIR`: The directory containing the trained model.
- `USE_SAFE_LIST=True`: Allow only the species, tissues, and cell types defined in the configured safe list. Replace the example safe list with one appropriate for the target tissue, or set this to `False`.
- `BATCH_SIZE`: Reduce this to 1024, 512, or lower if GPU memory is insufficient.
- The three `*_KEY` settings: The ground-truth label columns in the input data's `adata.obs`. Prediction can still run without ground-truth labels; only accuracy evaluation will be skipped.

In [ ]:
# ===== Common configuration settings =====
PROJECT_DIR = Path.cwd()

INPUT_H5AD = Path("/path/to/input.h5ad")
OUTPUT_H5AD = PROJECT_DIR / "outputs/test_annotated.h5ad"
MODEL_DIR = PROJECT_DIR / "models/last_version_gml"

USE_SAFE_LIST = True
SAFE_LIST_PATH = PROJECT_DIR / "safe_list.json"

DEVICE = "auto"       # Recommended: keep auto; alternatively, use "cuda" or "cpu"
BATCH_SIZE = 2048       # Reduce if GPU memory is insufficient
SAVE_METRICS_JSON = True

CELL_TYPE_KEY = "cell_type_ontology_term_id"
TISSUE_KEY = "general_tissue"
SPECIES_KEY = "organism"

### Safe List Format Example and Explanation

The project's `safe_list.json` is a **format example** that illustrates how to restrict the species, tissues, and cell types that the model can output. It is not a data configuration suitable for every experiment; replace it with a safe list appropriate for the target tissue.

The contents of `safe_list.json` are shown below:

```json
{
  "global": {
    "species": ["Homo sapiens", "Mus musculus"],
    "tissue": ["brain", "lung", "heart"],
    "cell_type": ["CL:0000540", "CL:0000127", "CL:0000236"]
  },
  "hierarchy": {
    "Homo sapiens": {
      "tissue": {
        "brain": {
          "cell_type": ["CL:0000540", "CL:0000127"]
        },
        "lung": {
          "cell_type": ["CL:0000236"]
        }
      }
    },
    "Mus musculus": {
      "tissue": {
        "brain": {
          "cell_type": ["CL:0000540"]
        }
      }
    }
  }
}
```

Meaning of each section:

- `global.species`: Species that may be predicted across the entire dataset. The example allows only human and mouse.
- `global.tissue`: Tissues that may be predicted across the entire dataset. The example allows only brain, lung, and heart.
- `global.cell_type`: Cell types that may be predicted across the entire dataset, specified as Cell Ontology IDs.
- `hierarchy`: Describes the allowed combinations of species, tissues, and cell types.
- `hierarchy.Homo sapiens.tissue.brain.cell_type`: Indicates that, after the model predicts human and then brain tissue, the cell type must be selected from `CL:0000540` and `CL:0000127`.
- In the example, only brain tissue is defined for mouse. Therefore, under the hierarchical constraints, the tissue corresponding to mouse is restricted to `brain`.

The model applies the prediction constraints in the following order:

```text
species allowed by global
        ↓
global.tissue ∩ tissues allowed for the species in hierarchy
        ↓
global.cell_type ∩ cell types allowed for the species+tissue combination
```

When customizing the safe list, note the following:

1. Species and tissue names must exactly match the class dictionaries in the model directory, including capitalization and spaces.
2. Cell types should be specified using Cell Ontology IDs supported by the model, such as `CL:0000540`.
3. A safe list can only restrict the model's existing classes; it cannot enable the model to recognize new classes outside its training data.
4. If a label is not present in the model's class dictionary, `[SafeList][WARN]` is displayed at runtime and that label is not included in the constraints.
5. If a constraint list or its intersection is empty, the current implementation falls back to the highest-scoring unconstrained prediction for that task. Therefore, check the warnings in the run log.
6. If the data contains tissues or cell types outside the safe list, set `USE_SAFE_LIST = False` or create a new JSON file appropriate for the experiment.

To try the example file, change the Step 2 configuration to:

```python
SAFE_LIST_PATH = PROJECT_DIR / "safe_list.json"
```

## Step 3: Validate the Configuration

This cell checks only the files and device; it does not start model inference. Continue after the configuration check succeeds.

In [ ]:
if DEVICE == "auto":
    device = "cuda" if torch.cuda.is_available() else "cpu"
elif DEVICE in {"cuda", "cpu"}:
    device = DEVICE
else:
    raise ValueError('DEVICE must be "auto", "cuda", or "cpu".')

if device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError('No available GPU was detected in the current environment. Set DEVICE to "cpu" or contact your administrator.')

required_model_files = [
    "unicell_v1.best.pth",
    "celltype_dict.pk",
    "tissue_dict.pk",
    "species_dict.pk",
    "gene_names.pk",
]

problems = []
if not INPUT_H5AD.is_file():
    problems.append(f"Input file not found: {INPUT_H5AD}")
if INPUT_H5AD.suffix.lower() != ".h5ad":
    problems.append("The input file must have the .h5ad extension")
if INPUT_H5AD.resolve() == OUTPUT_H5AD.resolve():
    problems.append("The output file must differ from the input file to avoid overwriting the original data")
if not MODEL_DIR.is_dir():
    problems.append(f"Model directory not found: {MODEL_DIR}")
else:
    for filename in required_model_files:
        if not (MODEL_DIR / filename).is_file():
            problems.append(f"Missing from the model directory: {filename}")
if USE_SAFE_LIST and not SAFE_LIST_PATH.is_file():
    problems.append(f"Safe list not found: {SAFE_LIST_PATH}")
if BATCH_SIZE <= 0:
    problems.append("BATCH_SIZE must be greater than 0")

if problems:
    raise FileNotFoundError("Configuration validation failed:\n- " + "\n- ".join(problems))

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
safe_list_path = str(SAFE_LIST_PATH) if USE_SAFE_LIST else None

print("Configuration validation passed")
print(f"Input file: {INPUT_H5AD}")
print(f"Model directory: {MODEL_DIR}")
print(f"Output file: {OUTPUT_H5AD}")
print(f"Device: {device}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Use safe list: {USE_SAFE_LIST}")

## Step 4: Read the Single-Cell Data

This step displays the number of cells, number of genes, and some sample information. Reading a large dataset may take some time.

In [ ]:
load_start = time.perf_counter()
adata = ad.read_h5ad(INPUT_H5AD)
load_seconds = time.perf_counter() - load_start

print(f"Data loaded in {load_seconds:.1f} seconds")
print(f"Cell count: {adata.n_obs:,}")
print(f"Gene count: {adata.n_vars:,}")
print(f"Expression matrix type: {type(adata.X).__name__}")
print("First 20 fields in obs:")
print(list(adata.obs.columns[:20]))

adata

## Step 5: Run UniCell Annotation

This is the most time-consuming step. Do not close the notebook or restart the kernel while it is running. If GPU memory is insufficient, return to Step 2, reduce `BATCH_SIZE`, restart the kernel, and rerun the notebook from the beginning.

In [ ]:
inference_start = time.perf_counter()

sc_dataset = unicell_predict(
    adata=adata,
    filepath=str(INPUT_H5AD),
    ckpt_dir=str(MODEL_DIR),
    batch_size=BATCH_SIZE,
    device=device,
    cell_type_key=CELL_TYPE_KEY,
    tissue_key=TISSUE_KEY,
    species_key=SPECIES_KEY,
    compute_metrics=False,
    safe_list_path=safe_list_path,
)

result_adata = sc_dataset.adata
inference_seconds = time.perf_counter() - inference_start
print(f"UniCell annotation completed in {inference_seconds / 60:.1f} minutes")

## Step 6: View the Prediction Results

The main prediction results for each cell are stored in `result_adata.obs`:

- `predicted_cell_type_ontology_id`: Predicted Cell Ontology ID
- `predicted_tissue`: Predicted tissue
- `predicted_species`: Predicted species

The following cells display the first 10 cells and the 20 most common predicted cell types, respectively.

In [ ]:
prediction_columns = [
    "predicted_cell_type_ontology_id",
    "predicted_tissue",
    "predicted_species",
]
available_prediction_columns = [
    column for column in prediction_columns if column in result_adata.obs.columns
]

display(result_adata.obs[available_prediction_columns].head(10))

if "predicted_cell_type_ontology_id" in result_adata.obs.columns:
    cell_type_summary = (
        result_adata.obs["predicted_cell_type_ontology_id"]
        .astype(str)
        .value_counts()
        .rename_axis("Cell Ontology ID")
        .reset_index(name="Cell count")
    )
    cell_type_summary["Percentage (%)"] = (
        cell_type_summary["Cell count"] / result_adata.n_obs * 100
    ).round(2)
    display(cell_type_summary.head(20))

## Step 7: Evaluate Accuracy (Optional)

If the input data contains ground-truth labels, Accuracy and Macro-F1 are reported; evaluation is skipped automatically when ground-truth labels are absent. Macro-F1 gives relatively balanced weight to common and rare classes.

In [ ]:
UNKNOWN_TOKENS = {"unknown", "nan", "None", "NA", "N/A", ""}

def evaluate_task(obs, true_key, predicted_key, task_name):
    if true_key not in obs.columns:
        print(f"{task_name}: Ground-truth label column '{true_key}' is missing; skipping evaluation.")
        return None
    if predicted_key not in obs.columns:
        print(f"{task_name}: Prediction column '{predicted_key}' is missing; skipping evaluation.")
        return None

    pairs = [
        (str(true_value), str(predicted_value))
        for true_value, predicted_value in zip(obs[true_key], obs[predicted_key])
        if str(true_value) not in UNKNOWN_TOKENS
        and str(predicted_value) not in UNKNOWN_TOKENS
    ]
    if not pairs:
        print(f"{task_name}: No valid labels are available for evaluation; skipping.")
        return None

    y_true, y_pred = zip(*pairs)
    raw_metrics = compute_metrics_cls(list(y_true), list(y_pred))
    result = {
        "n": len(pairs),
        "accuracy": float(raw_metrics.get("accuracy", np.nan)),
        "macro_f1": float(raw_metrics.get("macro_f1", np.nan)),
    }
    print(
        f"{task_name}: n={result['n']:,}, "
        f"Accuracy={result['accuracy']:.4f}, Macro-F1={result['macro_f1']:.4f}"
    )
    return result

metrics = {
    "cell_type": evaluate_task(
        result_adata.obs, CELL_TYPE_KEY,
        "predicted_cell_type_ontology_id", "Cell type"
    ),
    "tissue": evaluate_task(
        result_adata.obs, TISSUE_KEY, "predicted_tissue", "Tissue"
    ),
    "species": evaluate_task(
        result_adata.obs, SPECIES_KEY, "predicted_species", "Species"
    ),
}

metric_rows = [
    {"Task": task_name, **values}
    for task_name, values in metrics.items()
    if values is not None
]
if metric_rows:
    display(pd.DataFrame(metric_rows))

## Step 8: Save the Results

The saved data includes the original data, predicted labels, UniCell embeddings, model outputs for each classification task, and ontology hierarchy fields. Writing a large dataset to disk may take some time.

In [ ]:
save_start = time.perf_counter()
result_adata.write_h5ad(OUTPUT_H5AD)
save_seconds = time.perf_counter() - save_start
print(f"Annotation results saved to: {OUTPUT_H5AD}")
print(f"Write time: {save_seconds:.1f} seconds")

if SAVE_METRICS_JSON:
    metrics_path = OUTPUT_H5AD.with_name(OUTPUT_H5AD.stem + "_metrics.json")
    with metrics_path.open("w", encoding="utf-8") as file:
        json.dump(metrics, file, indent=2, ensure_ascii=False)
    print(f"Evaluation results saved to: {metrics_path}")

## Commonly Used Fields in the Output File

| Location | Field | Meaning |
|---|---|---|
| `obs` | `predicted_cell_type_ontology_id` | Predicted Cell Ontology ID for each cell |
| `obs` | `predicted_tissue` | Predicted tissue for each cell |
| `obs` | `predicted_species` | Predicted species for each cell |
| `obs` | `level_0`, `level_1`, ... | Hierarchical path of the predicted cell type in the ontology |
| `obsm` | `unicell_emb` | Numerical UniCell representation of each cell, suitable for downstream analysis |
| `obsm` | `cls_emb` | Raw model scores for cell-type classification |
| `obsm` | `tissue_cls_emb` | Raw model scores for tissue classification |
| `obsm` | `species_cls_emb` | Raw model scores for species classification |
| `uns` | `cls_cell_type`, `cls_tissue`, `cls_species` | Classes corresponding to each column of the model scores |

> For most biological analyses, use the predicted labels in `obs`. The raw model scores in `obsm` are intended mainly for further computational analysis.